In [120]:
EF434469_Repeat = EF434477_Repeat = EF434478_Repeat = "GTTTTTGTACTCTCAAGATTTAAGTAACTGTACAAC"
EF434469_TerminalRepeat = EF434477_TerminalRepeat = EF434478_TerminalRepeat = "GTTTTTGTACTCTCAAGATTTAAGTAACTGTACAGT"
EF434469_KnownRepeats = [
    37, 103, 169, 235, 301, 367, 433, 499,
    565, 631, 697, 763, 829, 895, 960, 1026,
    1092, 1158, 1224, 1290, 1356, 1422, 1488, 1554,
    1620, 1686, 1752, 1818, 1884, 1950, 2016, 2082,
    2148
]

In [121]:
from pathlib import Path
import numpy as np
import time
import tracemalloc

In [122]:
def metrics(reference, predicted, accession):
    length = len(accession[1])
    reference = set(reference)
    predicted = set(predicted)
    #print(reference == predicted)
    precision = recall = 0
    precision = len(predicted.intersection(reference)) / len(predicted)
    recall = len(predicted.intersection(reference)) / len(reference)
    jaccard = len(predicted.intersection(reference)) / len(predicted.union(reference))
    f1 = (2 * precision * recall) / (precision + recall)
    print("Length in Bases: " + str(length))
    print("Precision: " + str(precision * 100) + "%")
    print("Recall: " + str(recall * 100) + "%")
    print("Jaccard Similarity: " + str(jaccard * 100) + "%")
    print("F1 Score: " + str(round(f1 * 100, 2)) + "%")
    print("")


In [123]:
# ===== Preprocessing  =====

short_data = Path("short_data")
for path in short_data.glob("*.fasta"):
        #print(path)
        with open(path, 'r') as file:
                text = file.read()
                text = text.removeprefix('>')
                text = text.splitlines()
                metadata = text[0]
                sequence = text[1:]
                sequence = "".join(sequence)
                globals()[path.stem] = [metadata, sequence]
                globals()[path.stem + "_KnownRepeats"] = []
                for i in range(len(sequence)):
                        if (sequence[i:i+len(globals()[path.stem + "_Repeat"])] == globals()[path.stem + "_Repeat"]):
                                globals()[path.stem + "_KnownRepeats"].append(i)
                        elif (sequence[i:i+len(globals()[path.stem + "_TerminalRepeat"])] == globals()[path.stem + "_TerminalRepeat"]):
                                globals()[path.stem + "_KnownRepeats"].append(i)
                print(globals()[path.stem + "_Repeat"])
                print(globals()[path.stem + "_KnownRepeats"])


#EF434469.insert(0, EF434469_Repeat)
#EF434469.insert(0, EF434469_TerminalRepeat)
print((EF434469))
                

GTTTTTGTACTCTCAAGATTTAAGTAACTGTACAAC
[37, 103, 169, 235, 301, 367, 433, 499, 565, 631, 697, 763, 829, 895, 961, 1027, 1092, 1158, 1224, 1290, 1356, 1422, 1488, 1554, 1620, 1686, 1752, 1818, 1884, 1950, 2016, 2082, 2148, 2214, 2280]
GTTTTTGTACTCTCAAGATTTAAGTAACTGTACAAC
[37, 103, 169, 235, 301, 367, 433, 499, 565, 631, 697, 763, 829, 895, 961, 1027, 1093, 1159, 1224, 1290, 1356, 1422, 1488, 1554, 1620, 1686, 1752, 1818, 1884, 1950, 2016, 2082, 2148, 2214, 2280, 2346, 2412]
GTTTTTGTACTCTCAAGATTTAAGTAACTGTACAAC
[37, 103, 169, 235, 301, 367, 433, 499, 565, 631, 697, 763, 829, 895, 960, 1026, 1092, 1158, 1224, 1290, 1356, 1422, 1488, 1554, 1620, 1686, 1752, 1818, 1884, 1950, 2016, 2082, 2148]
['EF434469.1 Streptococcus thermophilus strain DGCC7710 CRISPR1 locus genomic sequence', 'ACTATGTGGGTATAAAAACGTCAAAATTTCATTTGAGGTTTTTGTACTCTCAAGATTTAAGTAACTGTACAACTGTTTGACAGCAAATCAAGATTCGAATTGTGTTTTTGTACTCTCAAGATTTAAGTAACTGTACAACAATGACGAGGAGCTATTGGCACAACTTACAGTTTTTGTACTCTCAAGATTTAAGTAACTGTACAACCGATTTGAC

In [124]:
# ===== Exact Matching Method: K-mer Indexing  =====

def kmerindexing(dataset, repeat, terminalrepeat, k):
    metadata, sequence = dataset[0:2]
    if (k > len(repeat)):
        raise ValueError("K exceeds repeat length")
    candidates = {repeat[0:k] : []}
    prediction = []
    count = 0
    for i in range(len(sequence) - k + 1):
        if (sequence[i:i+k] == repeat[0:k]):
            #print("Repeat at: " + str(i))
            candidates[repeat[0:k]].append(i)
    for index, position in enumerate(candidates[repeat[0:k]]):
        if (terminalrepeat == sequence[position:position+len((terminalrepeat))]):
            #print("Found TERMINAL @ " + str(position))
            prediction.append(position)
        if (repeat == sequence[position:position+len(repeat)]):
            #print("Found @" + str(position))
            prediction.append(position)
            count += 1
    return count, prediction
    #return candidates, len(candidates[repeat[0:k]])

#print(kmerindexing(EF434469, 3))
print("Runtime & Space for EF434469")
tracemalloc.start()
start = time.perf_counter()
EF434469_count, EF434469_prediction = kmerindexing(EF434469, EF434469_Repeat, EF434469_TerminalRepeat, 3)
elapsed = time.perf_counter() - start
_, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"Time: {elapsed:.6f} seconds")
print(f"Peak Python memory: {peak / 1024:.1f} KiB \n")

print("Runtime & Space for EF434477")
tracemalloc.start()
start = time.perf_counter()
EF434477_count, EF434477_prediction = kmerindexing(EF434477, EF434477_Repeat, EF434477_TerminalRepeat, 3)
elapsed = time.perf_counter() - start
_, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"Time: {elapsed:.6f} seconds")
print(f"Peak Python memory: {peak / 1024:.1f} KiB \n")

print("Runtime & Space for EF434478")
tracemalloc.start()
start = time.perf_counter()
EF434478_count, EF434478_prediction = kmerindexing(EF434478, EF434478_Repeat, EF434478_TerminalRepeat, 3)
elapsed = time.perf_counter() - start
_, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"Time: {elapsed:.6f} seconds")
print(f"Peak Python memory: {peak / 1024:.1f} KiB \n")

#print(EF434469_count)
#print(EF434469_prediction)
print("Metrics for EF434469")
metrics(EF434469_KnownRepeats, EF434469_prediction, EF434469)

print("Metrics for EF434477")
metrics(EF434477_KnownRepeats, EF434477_prediction, EF434477)

print("Metrics for EF434478")
metrics(EF434478_KnownRepeats, EF434478_prediction, EF434478)

#kmerindexing(EF434469, len(EF434469[0]))

    


Runtime & Space for EF434469
Time: 0.003586 seconds
Peak Python memory: 12.6 KiB 

Runtime & Space for EF434477
Time: 0.003978 seconds
Peak Python memory: 12.2 KiB 

Runtime & Space for EF434478
Time: 0.003935 seconds
Peak Python memory: 12.7 KiB 

Metrics for EF434469
Length in Bases: 2225
Precision: 100.0%
Recall: 100.0%
Jaccard Similarity: 100.0%
F1 Score: 100.0%

Metrics for EF434477
Length in Bases: 2489
Precision: 100.0%
Recall: 100.0%
Jaccard Similarity: 100.0%
F1 Score: 100.0%

Metrics for EF434478
Length in Bases: 2357
Precision: 100.0%
Recall: 100.0%
Jaccard Similarity: 100.0%
F1 Score: 100.0%



In [125]:
# ===== Approximate Matching Method: Hamming Distance  =====

def hamming_distance(dataset, repeat, terminalrepeat, tolerance=10):
    metadata, sequence = dataset[0:2]
    prediction = []
    n = len(sequence)
    m_repeat = len(repeat)
    m_terminal = len(terminalrepeat)
    repeat_threshold = tolerance/100 * m_repeat
    terminal_threshold = tolerance/100 * m_terminal
    for p in range(n):
        window_repeat = sequence[p:p+m_repeat]
        window_terminal = sequence[p:p+m_terminal]
        if p + m_repeat <= n:
            repeat_count = 0
            for i in range(m_repeat):
                if repeat[i] != window_repeat[i]:
                    repeat_count += 1
            if repeat_count <= repeat_threshold:
                prediction.append(p)
                continue
        if p + m_terminal <= n:
            terminal_count = 0
            for i in range(m_terminal):
                if terminalrepeat[i] != window_terminal[i]:
                    terminal_count += 1
            if terminal_count <= terminal_threshold:
                prediction.append(p)
                continue
    return len(prediction), prediction

print("Runtime & Space for EF434469")
tracemalloc.start()
start = time.perf_counter()
EF434469_count, EF434469_prediction = hamming_distance(EF434469, EF434469_Repeat, EF434469_TerminalRepeat, 50)
lapsed = time.perf_counter() - start
_, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"Time: {lapsed:.6f} seconds")
print(f"Peak Python memory: {peak / 1024:.1f} KiB \n")

print("Runtime & Space for EF434477")
tracemalloc.start()
start = time.perf_counter()
EF434477_count, EF434477_prediction = hamming_distance(EF434477, EF434477_Repeat, EF434477_TerminalRepeat, 50)
lapsed = time.perf_counter() - start
_, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"Time: {lapsed:.6f} seconds")
print(f"Peak Python memory: {peak / 1024:.1f} KiB \n")

print("Runtime & Space for EF434478")
tracemalloc.start()
start = time.perf_counter()
EF434478_count, EF434478_prediction = hamming_distance(EF434478, EF434478_Repeat, EF434478_TerminalRepeat, 50)
lapsed = time.perf_counter() - start
_, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"Time: {lapsed:.6f} seconds")
print(f"Peak Python memory: {peak / 1024:.1f} KiB \n")

print("Metrics for EF434469")
metrics(EF434469_KnownRepeats, EF434469_prediction, EF434469)

print("Metrics for EF434477")
metrics(EF434477_KnownRepeats, EF434477_prediction, EF434477)

print("Metrics for EF434478")
metrics(EF434478_KnownRepeats, EF434478_prediction, EF434478)



Runtime & Space for EF434469
Time: 0.019471 seconds
Peak Python memory: 13.2 KiB 

Runtime & Space for EF434477
Time: 0.021577 seconds
Peak Python memory: 13.0 KiB 

Runtime & Space for EF434478
Time: 0.020192 seconds
Peak Python memory: 12.3 KiB 

Metrics for EF434469
Length in Bases: 2225
Precision: 86.8421052631579%
Recall: 100.0%
Jaccard Similarity: 86.8421052631579%
F1 Score: 92.96%

Metrics for EF434477
Length in Bases: 2489
Precision: 86.04651162790698%
Recall: 100.0%
Jaccard Similarity: 86.04651162790698%
F1 Score: 92.5%

Metrics for EF434478
Length in Bases: 2357
Precision: 87.5%
Recall: 100.0%
Jaccard Similarity: 87.5%
F1 Score: 93.33%

